In [0]:
%run ./01_logging_utils.py

In [0]:
from pyspark.sql import functions as F, Window

silver = spark.table(T_SILVER)
silver_rows = silver.count()
if silver_rows == 0:
    raise Exception("Silver is empty. Run 04_silver_transform first.")
if spark.table(T_BRIDGE).count() == 0:
    raise Exception("Dimensions missing. Run 09_gold_dimensions first.")
print("Silver rows:", silver_rows)

In [0]:
w  = Window.partitionBy("coin_id").orderBy("event_ts")
wc = Window.partitionBy("coin_id")

h = (silver
     .withColumn("prev_price",  F.lag("price").over(w))
     .withColumn("prev_volume", F.lag("total_volume").over(w))
     .withColumn("avg_vol", F.avg("total_volume").over(wc))
     .withColumn("sd_vol",  F.stddev("total_volume").over(wc))
     .select(
        F.xxhash64("coin_id").alias("coin_key"),
        "event_ts",
        F.date_format("event_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("event_hour").alias("hour_key"),
        "price", "market_cap", "total_volume", "prev_price",
        F.when(F.col("prev_price") > 0,
               (F.col("price") - F.col("prev_price")) / F.col("prev_price") * 100
        ).alias("hourly_return_pct"),
        F.when(F.col("prev_volume") > 0,
               (F.col("total_volume") - F.col("prev_volume")) / F.col("prev_volume") * 100
        ).alias("volume_change_pct"),
        F.coalesce(F.col("total_volume") > F.col("avg_vol") + 2 * F.col("sd_vol"),
                   F.lit(False)).alias("is_volume_spike"),
        F.when(F.col("market_cap") > 0,
               F.col("total_volume") / F.col("market_cap")).alias("volume_to_mcap_ratio"),
        "batch_id",
        F.current_timestamp().alias("load_timestamp")))

run_gold_merge("fact_market_hourly", h, T_FACT_HOURLY, ["coin_key", "event_ts"],
               exact=["date_key", "hour_key", "is_volume_spike"],
               approx=["price", "market_cap", "total_volume", "prev_price", "hourly_return_pct",
                       "volume_change_pct", "volume_to_mcap_ratio"],
               extra=f"| silver_rows_read={silver_rows}")

In [0]:
d = (h.groupBy("coin_key", "date_key").agg(
        F.min_by("price", "event_ts").alias("open_price"),
        F.max_by("price", "event_ts").alias("close_price"),
        F.max("price").alias("high_price"),
        F.min("price").alias("low_price"),
        F.avg("price").alias("avg_price"),
        F.avg("market_cap").alias("avg_market_cap"),
        F.avg("total_volume").alias("avg_volume"),
        F.stddev("hourly_return_pct").alias("volatility_pct"),
        F.count("*").alias("hours_count"))
     .withColumn("daily_return_pct",
                 F.when(F.col("open_price") > 0,
                        (F.col("close_price") - F.col("open_price")) / F.col("open_price") * 100))
     .withColumn("intraday_range_pct",
                 F.when(F.col("open_price") > 0,
                        (F.col("high_price") - F.col("low_price")) / F.col("open_price") * 100))
     .withColumn("load_timestamp", F.current_timestamp()))

run_gold_merge("fact_coin_daily", d, T_FACT_DAILY, ["coin_key", "date_key"],
               exact=["hours_count"],
               approx=["open_price", "close_price", "high_price", "low_price", "avg_price",
                       "avg_market_cap", "avg_volume", "daily_return_pct",
                       "intraday_range_pct", "volatility_pct"],
               extra=f"| silver_rows_read={silver_rows}")

In [0]:
bridge = spark.table(T_BRIDGE)
dr = d.select("coin_key", "date_key", "daily_return_pct", "volatility_pct")

mkt = dr.groupBy("date_key").agg(F.avg("daily_return_pct").alias("mkt_ret"),
                                 F.avg("volatility_pct").alias("mkt_vol"))
med = mkt.agg(F.percentile_approx("mkt_vol", 0.5)).first()[0]
mkt = (mkt
       .withColumn("market_condition",
                   F.when(F.col("mkt_ret") > 1, "Bull")
                    .when(F.col("mkt_ret") < -1, "Bear")
                    .otherwise("Sideways"))
       .withColumn("is_high_volatility",
                   F.lit(False) if med is None
                   else F.coalesce(F.col("mkt_vol") > F.lit(float(med)), F.lit(False))))

pf = (bridge.join(dr.filter(F.col("daily_return_pct").isNotNull()), "coin_key")
      .groupBy("scenario_key", "date_key")
      .agg((F.sum(F.col("weight") * F.col("daily_return_pct")) / F.sum("weight"))
           .alias("portfolio_return_pct"))
      .join(mkt.select("date_key", "market_condition", "is_high_volatility"), "date_key")
      .withColumn("load_timestamp", F.current_timestamp()))

run_gold_merge("fact_portfolio_daily", pf, T_FACT_PORTF, ["scenario_key", "date_key"],
               exact=["market_condition", "is_high_volatility"],
               approx=["portfolio_return_pct"],
               extra=f"| silver_rows_read={silver_rows}")

In [0]:
for t in [T_FACT_HOURLY, T_FACT_DAILY, T_FACT_PORTF]:
    print(t.split(".")[-1], spark.table(t).count())
display(spark.sql(f"SELECT * FROM {T_LOGS} WHERE layer='gold' ORDER BY start_time DESC LIMIT 6"))

In [0]:
d.explain("formatted")